In [ ]:

import time
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, clear_output

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [ ]:
#Load the pharmacy sales dataset and prepare key business metrics
df = pd.read_csv("global_pharmacy_sales_2020_2025_daily_dataset.csv")

# Convert date column to datetime and sort chronologically
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Compute Total Revenue = Units Sold * Unit Price
df["revenue"] = df["units_sold"] * df["unit_price"]

print(f"Dataset loaded successfully: {df.shape[0]:,} rows and {df.shape[1]} columns.")
df.head()

Dataset loaded successfully: 177,990 rows and 15 columns.


,date,year,month,day,region,country,category,medicine,age_group,units_sold,unit_price,stock_level,expiry_days_remaining,covid_flag,revenue
0,2020-01-01,2020,1,1,Africa,Nigeria,Cough_Cold,Cough Syrup,65+,687,8.71,4287,113,1,5983.77
1,2020-01-01,2020,1,1,East Asia,South Korea,Chronic,Amlodipine,13-25,684,7.12,4481,408,1,4870.08
2,2020-01-01,2020,1,1,South America,Argentina,Chronic,Amlodipine,46-65,967,21.75,3964,498,1,21032.25
3,2020-01-01,2020,1,1,East Asia,China,Antibiotic,Azithromycin,26-45,545,5.22,4525,450,1,2844.90
4,2020-01-01,2020,1,1,North America,Canada,Antipyretic,Paracetamol,13-25,497,12.70,4598,560,1,6311.90


In [ ]:
#  Define the comprehensive BI Dashboard generator function
def create_bi_dashboard(data_slice, stream_status="Live Snapshot"):
    """
    Builds an interactive multi-visualization BI dashboard using Plotly subplots.
    - Row 1: 4 Executive KPI Indicators
    - Row 2: Live Revenue Trend & Category Performance Bar Chart
    - Row 3: Regional Market Share Donut Chart & Inventory Risk Matrix
    """
    # Calculate core real-time metrics
    total_rev = data_slice["revenue"].sum()
    total_units = data_slice["units_sold"].sum()
    avg_price = data_slice["unit_price"].mean()
    near_expiry_count = int((data_slice["expiry_days_remaining"] < 90).sum())

    # Initialize subplot canvas
    fig = make_subplots(
        rows=3, cols=4,
        specs=[
            [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
            [{"type": "scatter", "colspan": 2}, None, {"type": "bar", "colspan": 2}, None],
            [{"type": "domain", "colspan": 2}, None, {"type": "scatter", "colspan": 2}, None]
        ],
        subplot_titles=(
            "", "", "", "",
            "<b>Real-Time Revenue Trend ($)</b>",
            "<b>Revenue by Product Category</b>",
            "<b>Regional Market Share</b>",
            "<b>Inventory Risk Matrix (Stock vs Expiry Days)</b>"
        ),
        vertical_spacing=0.12,
        horizontal_spacing=0.08
    )

    # 1. Executive KPI Cards
    fig.add_trace(go.Indicator(
        mode="number",
        value=total_rev,
        number={"prefix": "$", "valueformat": ",.0f", "font": {"color": "#1E3A8A", "size": 28}},
        title={"text": "<b>Total Revenue</b>", "font": {"size": 13, "color": "#475569"}}
    ), row=1, col=1)

    fig.add_trace(go.Indicator(
        mode="number",
        value=total_units,
        number={"valueformat": ",.0f", "font": {"color": "#065F46", "size": 28}},
        title={"text": "<b>Total Units Sold</b>", "font": {"size": 13, "color": "#475569"}}
    ), row=1, col=2)

    fig.add_trace(go.Indicator(
        mode="number",
        value=avg_price,
        number={"prefix": "$", "valueformat": ".2f", "font": {"color": "#92400E", "size": 28}},
        title={"text": "<b>Avg Unit Price</b>", "font": {"size": 13, "color": "#475569"}}
    ), row=1, col=3)

    fig.add_trace(go.Indicator(
        mode="number",
        value=near_expiry_count,
        number={"valueformat": ",.0f", "font": {"color": "#991B1B", "size": 28}},
        title={"text": "<b>Near Expiry (<90 Days)</b>", "font": {"size": 13, "color": "#475569"}}
    ), row=1, col=4)

    # 2. Live Revenue Trend Line
    daily_trend = data_slice.groupby("date")["revenue"].sum().reset_index().tail(30)
    fig.add_trace(go.Scatter(
        x=daily_trend["date"],
        y=daily_trend["revenue"],
        mode="lines+markers",
        name="Revenue Trend",
        line=dict(color="#2563EB", width=2.5, shape="spline"),
        marker=dict(size=5, color="#1D4ED8"),
        fill="tozeroy",
        fillcolor="rgba(37, 99, 235, 0.12)",
        hovertemplate="<b>Date:</b> %{x|%Y-%m-%d}<br><b>Revenue:</b> $%{y:,.0f}<extra></extra>"
    ), row=2, col=1)

    # 3. Category Performance Bar Chart
    cat_perf = data_slice.groupby("category")["revenue"].sum().reset_index().sort_values("revenue", ascending=True)
    cat_colors = ["#60A5FA", "#34D399", "#FBBF24", "#F87171", "#A78BFA"]
    fig.add_trace(go.Bar(
        x=cat_perf["revenue"],
        y=cat_perf["category"],
        orientation="h",
        marker=dict(color=cat_colors[:len(cat_perf)]),
        text=cat_perf["revenue"].apply(lambda v: f"${v:,.0f}"),
        textposition="auto",
        name="Categories",
        hovertemplate="<b>%{y}:</b> $%{x:,.0f}<extra></extra>"
    ), row=2, col=3)

    # 4. Regional Market Share Donut Chart
    reg_perf = data_slice.groupby("region")["revenue"].sum().reset_index()
    fig.add_trace(go.Pie(
        labels=reg_perf["region"],
        values=reg_perf["revenue"],
        hole=0.45,
        marker=dict(colors=["#3B82F6", "#10B981", "#F59E0B", "#EF4444", "#8B5CF6", "#EC4899", "#14B8A6", "#6366F1"]),
        textinfo="percent+label",
        name="Regions",
        hovertemplate="<b>%{label}</b><br>Revenue: $%{value:,.0f}<br>Share: %{percent}<extra></extra>"
    ), row=3, col=1)

    # 5. Inventory Risk Matrix Scatter Plot
    sample_risk = data_slice.tail(200)
    fig.add_trace(go.Scatter(
        x=sample_risk["expiry_days_remaining"],
        y=sample_risk["stock_level"],
        mode="markers",
        marker=dict(
            size=9,
            color=sample_risk["units_sold"],
            colorscale="Plasma",
            showscale=True,
            colorbar=dict(title="Units Sold", thickness=12, len=0.3, y=0.15, x=1.02)
        ),
        text=sample_risk["medicine"],
        name="Stock Health",
        hovertemplate="<b>Medicine:</b> %{text}<br><b>Days to Expiry:</b> %{x}<br><b>Stock:</b> %{y:,.0f}<extra></extra>"
    ), row=3, col=3)

    # Layout Styling and Theme
    fig.update_layout(
        title=dict(
            text=f"<b>Global Pharmacy Business Intelligence Dashboard</b> | {stream_status}",
            font=dict(size=18, color="#0F172A"),
            x=0.01,
            y=0.98
        ),
        template="plotly_white",
        height=880,
        showlegend=False,
        margin=dict(t=80, b=40, l=50, r=50)
    )

    # Axis configuration
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Revenue ($)", row=2, col=1)
    fig.update_xaxes(title_text="Revenue ($)", row=2, col=3)
    fig.update_xaxes(title_text="Days Until Expiry", row=3, col=3)
    fig.update_yaxes(title_text="Stock Level", row=3, col=3)

    return fig

print("Dashboard generator function successfully defined!")

Dashboard generator function successfully defined!


In [ ]:
# Display initial baseline interactive BI dashboard
baseline_data = df.head(5000)
dashboard_fig = create_bi_dashboard(baseline_data, stream_status="Initial Baseline (5,000 Transactions)")
dashboard_fig.show()

In [ ]:
#  Automating updates with live data streams for real-time business intelligence
# Simulates incoming stream of transactions, continuously refreshing all visuals and KPIs

TOTAL_STREAM_TICKS = 5  
NEW_BATCH_SIZE = 1000    
STREAM_INTERVAL = 1.0     

print("Starting automated live data stream ingestion...")
current_index = 5000

for tick in range(1, TOTAL_STREAM_TICKS + 1):
    current_index += NEW_BATCH_SIZE
    stream_slice = df.iloc[:current_index]
    
    status_text = f"LIVE STREAMING (Batch {tick}/{TOTAL_STREAM_TICKS} - {len(stream_slice):,} Transactions Ingested)"
    updated_dashboard = create_bi_dashboard(stream_slice, stream_status=status_text)
    
    # Refresh view smoothly in real-time
    clear_output(wait=True)
    updated_dashboard.show()
    
    print(f"[Live Stream Active] Step {tick}/{TOTAL_STREAM_TICKS} rendered successfully. Next stream tick in {STREAM_INTERVAL}s...")
    time.sleep(STREAM_INTERVAL)

print("Live stream ingestion completed!")

[Live Stream Active] Step 5/5 rendered successfully. Next stream tick in 1.0s...
Live stream ingestion completed!


In [ ]:
#Interactive Time-Lapse Dashboard with Built-in Automated Play/Pause Controls
# Enables automated playback or manual scrubbing through time-periods directly in Plotly

monthly_perf = (
    df.groupby(["year", "month", "category"], as_index=False)
    .agg({"revenue": "sum", "units_sold": "sum"})
)
monthly_perf["period"] = monthly_perf["year"].astype(str) + "-" + monthly_perf["month"].astype(str).str.zfill(2)
monthly_perf = monthly_perf.sort_values("period").reset_index(drop=True)

animated_fig = px.bar(
    monthly_perf,
    x="category",
    y="revenue",
    color="category",
    animation_frame="period",
    animation_group="category",
    range_y=[0, monthly_perf["revenue"].max() * 1.15],
    title="<b>Automated Live Time-Stream: Monthly Category Revenue Evolution</b>",
    labels={"revenue": "Revenue ($)", "category": "Category", "period": "Time Period"},
    color_discrete_sequence=["#2563EB", "#10B981", "#F59E0B", "#EF4444", "#8B5CF6"]
)

animated_fig.update_layout(
    template="plotly_white",
    height=550,
    showlegend=False
)

# Configure smooth playback speed
animated_fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 600
animated_fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300

animated_fig.show()

In [ ]:
# Export complete interactive BI dashboard to standalone HTML
# Can be opened directly in any web browser or shared with stakeholders
export_filename = "pharmacy_bi_dashboard.html"
dashboard_fig.write_html(export_filename)

print(f"Interactive dashboard exported successfully to: {export_filename}")

Interactive dashboard exported successfully to: pharmacy_bi_dashboard.html
